> Exploration notebook. The production version of this code lives in `src/tenk_rag/collect.py`.

# Workflow

* Ticker -> Look up CIK -> Fetch filing list -> Find latest 10-K -> Download document

## Setup

In [1]:
import os
from pathlib import Path
import requests

HEADERS = {"User-Agent": os.environ["SEC_USER_AGENT"]}

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Look Up CIK

In [2]:
# Downloads the SEC's full ticker-to-CIK mapping
resp = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS)
resp.raise_for_status()

ticker_to_cik = {item["ticker"]: item["cik_str"] for item in resp.json().values()}

cik = ticker_to_cik["AAPL"]
print(cik)

320193


## Find the Latest 10-K

In [3]:
# This endpoint needs a 10-digit CIK
url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
resp = requests.get(url, headers=HEADERS)
resp.raise_for_status()

recent = resp.json()["filings"]["recent"]
idx = recent["form"].index("10-K")

filing_date = recent["filingDate"][idx]
# Unique ID for each filing, used to build the document URL
accession = recent["accessionNumber"][idx]
primary_doc = recent["primaryDocument"][idx]
print(filing_date, accession, primary_doc)

2026-11-03 0000320193-26-000026 aapl-20260926.htm


## Download the Document

In [4]:
# The actual 10-K file URL
doc_url = (
	f"https://www.sec.gov/Archives/edgar/data/{cik}/"
	f"{accession.replace('-', '')}/{primary_doc}"
)
resp = requests.get(doc_url, headers=HEADERS)
resp.raise_for_status()

# Saves as TICKER_FORM_DATE.html, keeping files easy to identify later
out_path = DATA_DIR / f"AAPL_10-K_{filing_date}.html"
out_path.write_text(resp.text, encoding="utf-8")
print(out_path, f"{len(resp.text):,} characters")

../data/raw/AAPL_10-K_2026-11-03.html 1,650,108 characters


## Helper Functions

In [13]:
import json
import time

# Waits 0.2 seconds before each request to stay under the SEC's 10-requests-per-second
# limit. Keeping requests in one place ensures the rule is always followed.
def get_json(url):
	time.sleep(0.2)
	resp = requests.get(url, headers=HEADERS)
	resp.raise_for_status()
	return resp.json()

# Finds the latest 10-K in one batch of filings. Returns None instead of raising an
# error if none exists
def search_10k(batch):
	if "10-K" not in batch["form"]:
		return None
	i = batch["form"].index("10-K")
	return {
		"filing_date": batch["filingDate"][i],
		"accession": batch["accessionNumber"][i],
		"primary_doc": batch["primaryDocument"][i],
	}

# Searches recent first, then falls back to the older filing batches listed in files
def find_latest_10k(cik):
	filings = get_json(f"https://data.sec.gov/submissions/CIK{cik:010d}.json")["filings"]

	result = search_10k(filings["recent"])
	if result:
		return result

	for extra in filings.get("files", []):
		result = search_10k(get_json(f"https://data.sec.gov/submissions/{extra['name']}"))
		if result:
			return result

	return None


## Download Function

In [14]:
def download_latest_10k(ticker):
	cik = CIK_OVERRIDES.get(ticker, ticker_to_cik[ticker])
	filing = find_latest_10k(cik)
	if filing is None:   # if no 10-K was found
		print(f"{ticker}: no 10-K found")
		return None

	out_path = DATA_DIR / f"{ticker}_10-K_{filing['filing_date']}.html"
	if not out_path.exists():   # runs the download only if the file doesn't exist yet
		url = (   # builds the 10-K document URL
			f"https://www.sec.gov/Archives/edgar/data/{cik}/"
			f"{filing['accession'].replace('-', '')}/{filing['primary_doc']}"
		)
		time.sleep(0.2)
		resp = requests.get(url, headers=HEADERS)
		resp.raise_for_status()
		out_path.write_text(resp.text, encoding="utf-8")

	# **filing: unpacks the filing dictionary (date, accession, doc name) into the result
	return {"ticker": ticker, "cik": cik, **filing, "path": str(out_path)}

## Run and Save a Manifest

In [15]:
# Ticker now maps to a new holding company with no 10-K yet
# Use the prior registrant
CIK_OVERRIDES = {"XOM": 34088}
TICKERS = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN",
    "JPM", "BAC", "GS", "V",
    "JNJ", "PFE", "UNH",
    "KO", "PG", "WMT", "NKE",
    "XOM", "CVX", "CAT", "BA",
]

manifest = []	# A record which filing was downloaded for each company
				# Later used to cite sources in RAG answers
for ticker in TICKERS:
	info = download_latest_10k(ticker)
	if info:
		manifest.append(info)
		print(f"{ticker}: {info['filing_date']}")

# manifest.json: Saves the record to disk, which matters for reproducibility
(DATA_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\n{len(manifest)}/{len(TICKERS)} downloaded")

AAPL: 2026-11-03
MSFT: 2026-07-29
NVDA: 2026-02-25
GOOGL: 2026-02-05
AMZN: 2026-02-06
JPM: 2026-02-13
BAC: 2026-02-25
GS: 2026-02-25
V: 2026-11-05
JNJ: 2026-02-11
PFE: 2026-02-26
UNH: 2026-03-02
KO: 2026-02-20
PG: 2026-08-04
WMT: 2026-03-13
NKE: 2026-07-15
XOM: 2026-02-18
CVX: 2026-02-24
CAT: 2026-02-13
BA: 2026-01-30

20/20 downloaded


In [11]:
from collections import Counter

cik = ticker_to_cik["XOM"]
data = get_json(f"https://data.sec.gov/submissions/CIK{cik:010d}.json")

print("CIK:", cik)
print("Name:", data["name"])
# Counts filings by form type and shows the top 10, revealing whether a 10-K exists
# at all or under a different name
print("Forms:", Counter(data["filings"]["recent"]["form"]).most_common(10))
print("Extra files:", data["filings"].get("files"))

CIK: 2115436
Name: ExxonMobil Holdings Corp
Forms: [('S-8 POS', 23), ('8-K', 6), ('SCHEDULE 13G', 3), ('DEFA14A', 2), ('10-Q', 2), ('SCHEDULE 13G/A', 1), ('SD', 1), ('424B2', 1), ('FWP', 1), ('424B3', 1)]
Extra files: []
